# 05 — Tracking de Objetos

## ¿Qué vamos a construir hoy?

Procesarás un video real y seguirás cada persona con un número de identidad
que se mantiene estable aunque el objeto se mueva.

**Aprenderás a:**
- Entender la diferencia entre detectar objetos y seguirlos en el tiempo
- Usar `sv.ByteTrack` para asignar IDs persistentes
- Procesar video con `sv.process_video`

## Detección vs. Tracking

Detectar objetos en video es como revisar cada foto de una cámara de seguridad:
sabes cuántos objetos hay en ese instante, pero no sabes si el objeto
del frame 1 es el mismo que el del frame 2.

**El tracker resuelve eso:**
actúa como un guardia con lista de asistencia — compara los objetos nuevos
con los del frame anterior y asigna el mismo número de credencial si los reconoce.

```
Frame N:   detecciones sin ID  ──► tracker ──► detecciones con tracker_id
Frame N+1: detecciones sin ID  ──► tracker ──► mismos tracker_id (si mismo objeto)

In [3]:
!pip install supervision ultralytics



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import cv2
from ultralytics import YOLO
import supervision as sv
import numpy as np

model = YOLO("yolov8n.pt")
box_annotator = sv.BoxAnnotator()

# sv.process_video
source_path = ruta del video que queremos analizar
target_path = ruta del video por guardar
callback = que vamos a hacer con la imagen del frame (funcion)
show_progress = si queremos mostrar el progreso del video analizado

In [11]:
def inference_callback(frame: np.ndarray, _: int) -> np.ndarray: 
    results = model(frame)[0]
    detections = sv.Detections.from_ultralytics(results)
    return box_annotator.annotate(frame.copy(), detections = detections)

sv.process_video(
    source_path = 'people-walking.mp4',
    target_path = 'run-inference-result.mp4',
    callback = inference_callback,
    show_progress = True
)

Processing video:   0%|          | 0/341 [00:00<?, ?it/s]


0: 384x640 37 persons, 2 birds, 14.7ms
Speed: 8.3ms preprocess, 14.7ms inference, 3.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 38 persons, 3 birds, 4.7ms
Speed: 1.4ms preprocess, 4.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 3 birds, 5.1ms
Speed: 1.8ms preprocess, 5.1ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 37 persons, 3 birds, 5.1ms
Speed: 3.2ms preprocess, 5.1ms inference, 3.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 24.1ms
Speed: 3.0ms preprocess, 24.1ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 38 persons, 2 birds, 4.6ms
Speed: 4.9ms preprocess, 4.6ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 13.4ms
Speed: 1.4ms preprocess, 13.4ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 5.6ms
Speed: 1.4

# Use Tracking

In [13]:
tracker = sv.ByteTrack()
box_annotator_tracking = sv.BoxAnnotator(color = sv.Color.YELLOW, thickness = 5)

tracker.reset()

def tracking_callback(frame: np.ndarray, _: int) -> np.ndarray:
    results = model(frame)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = tracker.update_with_detections(detections)
    return box_annotator_tracking.annotate(frame.copy(), detections = detections)

sv.process_video(
    source_path = 'people-walking.mp4',
    target_path = 'tracking-people.mp4',
    callback = tracking_callback,
    show_progress = True
)


Processing video:   0%|          | 0/341 [00:00<?, ?it/s]


0: 384x640 37 persons, 2 birds, 10.0ms
Speed: 1.4ms preprocess, 10.0ms inference, 3.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 38 persons, 3 birds, 4.7ms
Speed: 1.5ms preprocess, 4.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 3 birds, 10.0ms
Speed: 1.3ms preprocess, 10.0ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 37 persons, 3 birds, 8.2ms
Speed: 1.3ms preprocess, 8.2ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 6.1ms
Speed: 1.2ms preprocess, 6.1ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 38 persons, 2 birds, 9.5ms
Speed: 1.3ms preprocess, 9.5ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 4.3ms
Speed: 1.6ms preprocess, 4.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 6.0ms
Speed: 1.1ms

## Tracer with IDs

In [17]:
from supervision import Color
box_annotator_ids = sv.BoxAnnotator(color = Color.BLACK, thickness = 7)
label_annotator_ids = sv.LabelAnnotator(color = Color.BLACK)
tracker.reset()

def tracking_ids_callback(frame: np.ndarray, _: int) -> np.ndarray:
    results = model(frame)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = tracker.update_with_detections(detections)

    labels = [
        f"#{tracker_id} {results.names[class_id]}"
        for class_id, tracker_id
        in zip(detections.class_id, detections.tracker_id)
    ]

    annotated = box_annotator_ids.annotate(scene = frame.copy(), detections = detections)
    return label_annotator_ids.annotate(scene = annotated, detections = detections, labels = labels)

sv.process_video(
    source_path = 'people-walking.mp4',
    target_path = 'tracking-ids-people.mp4', 
    callback = tracking_ids_callback,
    show_progress = True
)

Processing video:   0%|          | 0/341 [00:00<?, ?it/s]


0: 384x640 37 persons, 2 birds, 5.4ms
Speed: 2.1ms preprocess, 5.4ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 38 persons, 3 birds, 8.5ms
Speed: 1.4ms preprocess, 8.5ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 3 birds, 8.8ms
Speed: 1.3ms preprocess, 8.8ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 37 persons, 3 birds, 11.0ms
Speed: 2.0ms preprocess, 11.0ms inference, 4.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 5.2ms
Speed: 1.5ms preprocess, 5.2ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 38 persons, 2 birds, 4.4ms
Speed: 1.4ms preprocess, 4.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 5.6ms
Speed: 1.2ms preprocess, 5.6ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 8.8ms
Speed: 1.3ms p

## Trace Annotator

color: Color,
position: Position,
trace_length: int,
thickness: int,
color_lookup: str

In [18]:
from PIL import ImageColor
tracker.reset()
box_annotator_trace = sv.BoxAnnotator(Color.ROBOFLOW, thickness=6)
label_annotator = sv.LabelAnnotator(color=Color.ROBOFLOW)
trace_annotator = sv.TraceAnnotator(color = Color.ROBOFLOW)

def trace_callback(frame: np.ndarray, _: int) -> np.ndarray:
    results = model(frame)[0]
    detections = sv.Detections.from_ultralytics(results)
    detections = tracker.update_with_detections(detections)

    labels = [
        f"#{tracker_id} {results.names[class_id]}"
        for class_id, tracker_id
        in zip(detections.class_id, detections.tracker_id)
    ]

    annotated = box_annotator_trace.annotate(frame.copy(), detections= detections)
    annotated = label_annotator.annotate(annotated, detections = detections, labels = labels)
    return trace_annotator.annotate(annotated, detections = detections)

sv.process_video(
    source_path='people-walking.mp4',
    target_path='trace-people-walking.mp4',
    callback=trace_callback,
    show_progress=True
)

Processing video:   0%|          | 0/341 [00:00<?, ?it/s]


0: 384x640 37 persons, 2 birds, 5.4ms
Speed: 6.2ms preprocess, 5.4ms inference, 5.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 38 persons, 3 birds, 26.3ms
Speed: 2.1ms preprocess, 26.3ms inference, 8.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 3 birds, 4.8ms
Speed: 3.8ms preprocess, 4.8ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 37 persons, 3 birds, 8.2ms
Speed: 4.6ms preprocess, 8.2ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 9.3ms
Speed: 2.8ms preprocess, 9.3ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 38 persons, 2 birds, 6.3ms
Speed: 1.3ms preprocess, 6.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 6.4ms
Speed: 1.4ms preprocess, 6.4ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 6.1ms
Speed: 1.4ms p

## Heat Map Annotator

position: Position,
opacity: float
radius: int
kernel_size = int
top_hue: int
bottom_hue: int

In [ ]:
heat_map_annotator = sv.HeatMapAnnotator()
tracker.reset()

def heat_map_callback(frame: np.ndarray, _: int) -> np.ndarray:
    results = model(frame)[0] #me habia faltado agregar [0]
    detections = sv.Detections.from_ultralytics(results)
    detections = tracker.update_with_detections(detections)

    return heat_map_annotator.annotate(frame.copy(), detections = detections)

sv.process_video(
    source_path='people-walking.mp4',
    target_path = 'heat-people-walking.mp4',
    callback=heat_map_callback,
    show_progress=True
)

Processing video:   0%|          | 0/341 [00:00<?, ?it/s]


0: 384x640 37 persons, 2 birds, 31.8ms
Speed: 1.8ms preprocess, 31.8ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 38 persons, 3 birds, 17.5ms
Speed: 1.9ms preprocess, 17.5ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 3 birds, 19.8ms
Speed: 2.0ms preprocess, 19.8ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 37 persons, 3 birds, 16.0ms
Speed: 1.8ms preprocess, 16.0ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 9.9ms
Speed: 1.8ms preprocess, 9.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 38 persons, 2 birds, 7.4ms
Speed: 1.9ms preprocess, 7.4ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 7.9ms
Speed: 6.9ms preprocess, 7.9ms inference, 6.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 2 birds, 82.0ms
Speed: 